**Mini Project-3**

In [142]:
import cupy as cp
from google.colab import drive
# Mounting Google Drive to access data
drive.mount ('/content/drive')

# Function to load data from a file
def loading_of_data(file_path):
    try:
        # Convert space-separated values into a CuPy array
        with open(file_path, 'r') as file:
            # Use np.fromstring to convert space-separated values into an array
            data = cp.fromstring(file.read(), sep=' ')
            data = cp.asarray(data)
            # Reshape data if it's a single-dimensional array
            if len(data.shape) == 1:
                data = data.reshape(-1, 1)
        return data
    except ValueError as e:
        print(f"Error loading data from {file_path}: {e}")
        return None
    except FileNotFoundError:
        print(f"File not found: {file_path}")
        return None

# File paths for training and testing data
file_path = "/content/drive/MyDrive/"
X_train_path = file_path + 'X_train.txt'
y_train_path = file_path + 'y_train.txt'
X_test_path = file_path + 'X_test.txt'
y_test_path = file_path + 'y_test.txt'

# Load features and labels using the defined function
X_train = loading_of_data(X_train_path)
y_train = loading_of_data(y_train_path).astype(int)
X_test = loading_of_data(X_test_path)
y_test = loading_of_data(y_test_path).astype(int)

# Define the neural network architecture
X_train = X_train.reshape((7352, 561))
X_test = X_test.reshape((2947, 561))
input_size = X_train.shape[1]
output_size = len(cp.unique(y_train))


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [143]:
# Define hyperparameters for the neural network
learning_rate = 0.01              # Rate at which the model updates its parameters during training
N1 = 300                          # Number of neurons in the first hidden layer
N2 = 200                          # Number of neurons in the second hidden layer
momentum = 0.9                 # Momentum value for optimization, helps accelerate gradient descent in the relevant direction
batch_size = 50                  # Number of training examples utilized in one iteration of gradient descent
validation_percentage = 0.1       # Percentage of training set used for validation during training
num_epochs = 100                  # Number of times the entire dataset is passed forward and backward through the neural network


In [144]:
# Initialization of some variables to keep track of best accuracies, parameters, and other settings
best_train_accuracy = 0.0
best_test_accuracy = 0.0
best_params = None
dropout_rate = 0.5
early_stopping_patience = 10  # Adjust this value based on your preferences
validation_accuracies = []
# Define the percentage of the training set to use as a validation set
validation_percentage = 0.1

# Splitting the training set into training and validation sets based on the validation percentage
data_split = int((1 - validation_percentage) * len(X_train))
X_train_split, y_train_split = X_train[:data_split], y_train[:data_split]
X_val_split, y_val_split = X_train[data_split:], y_train[data_split:]

# Function to initialize neural network parameters
def initialization():
    params = {
        'W1': cp.random.uniform(-0.1, 0.1, (input_size, N1)),
        'b1': cp.zeros((1, N1)),
        'W2': cp.random.uniform(-0.1, 0.1, (N1, N2)),
        'b2': cp.zeros((1, N2)),
        'W3': cp.random.uniform(-0.1, 0.1, (N2, output_size)),
        'b3': cp.zeros((1, output_size))
    }
    return params

def calculate_accuracy(y_true, y_pred):
    y_true = y_true.flatten()
    predicted_labels = cp.argmax(y_pred, axis=1) # Convert predicted probabilities to class labels
    correct_predictions = cp.sum(predicted_labels == y_true) # Count correct predictions
    accuracy = correct_predictions / len(y_true) # Calculate accuracy
    return accuracy


    # Count correct predictions
    correct_predictions = cp.sum(predicted_labels == y_true)

    # Calculate accuracy
    accuracy = correct_predictions / len(y_true)

    return accuracy

# Define the ReLU activation function
def relu(x):
    return cp.maximum(0, x)

# Define the softmax activation function
def softmax(x):
    exp_x = cp.exp(x - cp.max(x, axis=1, keepdims=True))
    return exp_x / cp.sum(exp_x, axis=1, keepdims=True)

# Define the categorical cross-entropy loss
def categorical_cross_entropy(y_true, y_pred):
    epsilon = 1e-15
    y_pred = cp.clip(y_pred, epsilon, 1 - epsilon)
    return -cp.sum(y_true * cp.log(y_pred)) / len(y_true)

# Forward pass of the neural network
def forward_pass(X, params, is_training=True):
    # For Layer 1
    z1 = cp.dot(X, params['W1']) + params['b1']
    a1 = relu(z1)

    # For Layer 2
    z2 = cp.dot(a1, params['W2']) + params['b2']
    a2 = relu(z2)

    # Dropout in the second hidden layer
    if is_training:
        dropout_mask = (cp.random.rand(*a2.shape) < (1 - dropout_rate))
        a2 *= dropout_mask / (1 - dropout_rate)

    # Output layer
    z3 = cp.dot(a2, params['W3']) + params['b3']
    a3 = softmax(z3)

    return a1, a2, a3

# Backward pass of the neural network
def backward_pass(X, y, params, a1, a2, a3, momentum):
    m = len(X)
    # Calculate loss at output layer
    arr = cp.eye(output_size)[y]
    size = y.shape[0]
    reshaped_arr = arr.reshape((size, 6))
    dz3 = a3 - reshaped_arr

    # Calculate gradients
    dW3 = cp.dot(a2.T, dz3) / m
    db3 = cp.sum(dz3, axis=0, keepdims=True) / m

    # Layer 2
    dz2 = cp.dot(dz3, params['W3'].T) * (a2 > 0)
    dW2 = cp.dot(a1.T, dz2) / m
    db2 = cp.sum(dz2, axis=0, keepdims=True) / m

    # Layer 1
    dz1 = cp.dot(dz2, params['W2'].T) * (a1 > 0)
    dW1 = cp.dot(X.T, dz1) / m
    db1 = cp.sum(dz1, axis=0, keepdims=True) / m

    # Update weights and biases of the neural network based on calculated gradients
    params['W1'] -= learning_rate * dW1 + momentum * params['W1']
    params['b1'] -= learning_rate * db1 + momentum * params['b1']
    params['W2'] -= learning_rate * dW2 + momentum * params['W2']
    params['b2'] -= learning_rate * db2 + momentum * params['b2']
    params['W3'] -= learning_rate * dW3 + momentum * params['W3']
    params['b3'] -= learning_rate * db3 + momentum * params['b3']

    return params

In [145]:
# # Initialize parameters for the neural network
params = initialization()

# Training loop that iterates over the specified number of epochs
for epoch in range(num_epochs):
    # Iterating through the training data in mini-batches
    for i in range(0, len(X_train), batch_size):
        # Extract a mini-batch of data
        X_batch, y_batch = X_train[i:i+batch_size], y_train[i:i+batch_size]

        # Forward pass
        a1, a2, a3 = forward_pass(X_batch, params)

        # Backward pass
        params = backward_pass(X_batch, y_batch, params, a1, a2, a3, momentum)

    # Forward pass on the validation set after each epoch
    a1_val, a2_val, a3_val = forward_pass(X_val_split, params)

    # Calculate accuracy on the validation set
    val_accuracy = calculate_accuracy(y_val_split, a3_val)
    validation_accuracies.append(val_accuracy)

    # Print validation accuracy for the current epoch
    print(f"Epoch {epoch + 1}/{num_epochs}, Validation Accuracy: {val_accuracy}")

    # Check for early stopping
    if val_accuracy > best_test_accuracy:
        best_test_accuracy = val_accuracy
        early_stopping_patience = 10  # Reset patience when there's an improvement
        best_params = params
    else:
        early_stopping_patience -= 1               # Decrease patience if there's no improvement
        if early_stopping_patience == 0:           # If patience runs out, stop training
            print("Early stopping: No improvement in validation accuracy.")
            break
# if best_params is not None:
    a1_test, a2_test, a3_test = forward_pass(X_test, params)
    y_test_flat = y_test.flatten()
    test_accuracy = calculate_accuracy(y_test_flat, a3_test)
    print(f"Best Validation Accuracy: {best_test_accuracy}")
    print(f"Test Accuracy with Best Parameters: {test_accuracy}")

Epoch 1/100, Validation Accuracy: 0.1671195652173913
Best Validation Accuracy: 0.1671195652173913
Test Accuracy with Best Parameters: 0.15982354937224297
Epoch 2/100, Validation Accuracy: 0.1671195652173913
Best Validation Accuracy: 0.1671195652173913
Test Accuracy with Best Parameters: 0.15982354937224297
Epoch 3/100, Validation Accuracy: 0.1671195652173913
Best Validation Accuracy: 0.1671195652173913
Test Accuracy with Best Parameters: 0.15982354937224297
Epoch 4/100, Validation Accuracy: 0.1671195652173913
Best Validation Accuracy: 0.1671195652173913
Test Accuracy with Best Parameters: 0.15982354937224297
Epoch 5/100, Validation Accuracy: 0.1671195652173913
Best Validation Accuracy: 0.1671195652173913
Test Accuracy with Best Parameters: 0.15982354937224297
Epoch 6/100, Validation Accuracy: 0.1671195652173913
Best Validation Accuracy: 0.1671195652173913
Test Accuracy with Best Parameters: 0.15982354937224297
Epoch 7/100, Validation Accuracy: 0.1671195652173913
Best Validation Accurac

In [146]:
#FOR ONLINE LEARNING

# for epoch in range(num_epochs):
#     for i in range(len(X_train)):
#         X_sample, y_sample = X_train[i:i+1], y_train[i:i+1]

#         # Forward pass
#         a1, a2, a3 = forward_pass(X_sample, params)

#         # Backward pass
#         params = backward_pass(X_sample, y_sample, params, a1, a2, a3, momentum)

#         # Print accuracy for every N samples (adjust N based on your preference)
#         if i % 100 == 0:
#             a1_test, a2_test, a3_test = forward_pass(X_test, params)
#             y_test_flat = cp.ndarray.flatten(y_test)
#             test_accuracy = calculate_accuracy(y_test_flat, a3_test)
#             print(f"Sample {i}/{len(X_train)}, Test Accuracy: {test_accuracy}")

#     # Forward pass on test set after each epoch
#     a1_test, a2_test, a3_test = forward_pass(X_test, params)
#     y_test_flat = cp.ndarray.flatten(y_test)
#     test_accuracy = calculate_accuracy(y_test_flat, a3_test)
#     print(f"Best Validation Accuracy: {best_test_accuracy}")
#     print(f"Test Accuracy with Best Parameters: {test_accuracy}")
#     # Print accuracy after each epoch
#     print(f"Epoch {epoch + 1}/{num_epochs}, Test Accuracy: {test_accuracy}")

#     # Check for early stopping (optional)
#     if test_accuracy > best_test_accuracy:
#         best_test_accuracy = test_accuracy
#         early_stopping_patience = 3  # Reset patience when there's an improvement
#     else:
#         early_stopping_patience -= 1
#         if early_stopping_patience == 0:
#             print("Early stopping: No improvement in test accuracy.")
#             break